In [ ]:
# ======================== #import the package ========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
import zipfile
import os

# ======================== #name the variables ========================
source = (0, 0)
destination = (2, 2)

# ======================== #read the file and store data ========================

# 直接指向当前工作目录，CSV 就放在这里
corrected_extract_path = "."

# 读取所有 *_tree.csv 并配对对应的 *_building.csv，计算 avg_shadow 存到字典
shadow_profiles = {}
for fname in os.listdir(corrected_extract_path):
    if not fname.endswith("_tree.csv"):
        continue

    prefix    = fname[:-9]  # 去掉 "_tree.csv"，得到 "edge_00_01" 之类
    tree_file = os.path.join(corrected_extract_path, f"{prefix}_tree.csv")
    bld_file  = os.path.join(corrected_extract_path, f"{prefix}_building.csv")

    if not os.path.exists(bld_file):
        print(f"Warning: 找不到 {prefix}_building.csv，已跳过")
        continue

    df_tree   = pd.read_csv(tree_file, header=0)
    df_bld    = pd.read_csv(bld_file,  header=0)

    tree_vals = df_tree.select_dtypes(include=[np.number]).astype(float).values
    bld_vals  = df_bld.select_dtypes(include=[np.number]).astype(float).values

    combined   = np.where(bld_vals == 1.0, 1.0, tree_vals)
    avg_shadow = combined.mean(axis=1)

    shadow_profiles[prefix] = avg_shadow


# Update nested path based on structure inside the ZIP
corrected_extract_path = os.path.join(extract_path, "edge_shadow_data")


# ======================== #initialize for the graph ========================

# Define edge lengths and build the graph
edge_lengths = {
    ((0, 0), (0, 1)): 100, ((0, 1), (0, 2)): 120,
    ((1, 0), (1, 1)): 110, ((1, 1), (1, 2)): 130,
    ((2, 0), (2, 1)): 105, ((2, 1), (2, 2)): 115,
    ((0, 0), (1, 0)): 90,  ((1, 0), (2, 0)): 95,
    ((0, 1), (1, 1)): 85,  ((1, 1), (2, 1)): 100,
    ((0, 2), (1, 2)): 110, ((1, 2), (2, 2)): 105,
    ((0, 0), (1, 1)): 140, ((1, 1), (2, 2)): 145,
    ((0, 2), (1, 1)): 135, ((1, 1), (2, 0)): 150
}

graph = {}
for (i, j), dist in edge_lengths.items():
    graph.setdefault(i, {})[j] = dist
for (_, j) in edge_lengths.keys():
    if j not in graph:
        graph[j] = {}

# ======================== #initialize ========================

def is_dominated(obj1, obj2):
    less = False
    for a, b in zip(obj1, obj2):
        if a > b:
            return False
        elif a < b:
            less = True
    return less

def add_label_check(label_dict, node, new_obj):
    for existing_obj in label_dict[node]:
        if is_dominated(new_obj, existing_obj):
            return False
    return True

def load_shadow_profile(edge_name_prefix):
    tree_file = f"{corrected_extract_path}/{edge_name_prefix}_tree.csv"
    bld_file = f"{corrected_extract_path}/{edge_name_prefix}_building.csv"

    tree_shadow = pd.read_csv(tree_file, header=0)
    bld_shadow = pd.read_csv(bld_file, header=0)

    tree_vals = tree_shadow.select_dtypes(include=[np.number]).astype(float).values
    bld_vals = bld_shadow.select_dtypes(include=[np.number]).astype(float).values

    combined = np.where(bld_vals == 1.0, 1.0, tree_vals)
    avg_shadow = combined.mean(axis=1)  # average across subintervals per timestep

    return avg_shadow

def calculate_solar_exposure(i, j):
    edge_name = f"edge_{i[0]}{i[1]}_{j[0]}{j[1]}"
    shadow_profile = load_shadow_profile(edge_name)
    R = np.full(150, 1000)
    C = np.full(150, 0.2)
    W = (1 - C) * R * (1 - shadow_profile)
    return np.sum(W)

# ======================== #loop algorithm ========================

def bi_objective_label_correcting(graph, source, destination):
    queue = []
    heapq.heappush(queue, (0, 0, [source], [0, 0]))  # priority, id, path, [distance, solar]
    label_dict = {node: [] for node in graph}
    paths = []
    label_id = 0

    while queue:
        _, _, path, obj = heapq.heappop(queue)
        current_node = path[-1]

        if not add_label_check(label_dict, current_node, obj):
            continue

        label_dict[current_node].append(obj)

        if current_node == destination:
            paths.append({'path': path, 'objectives': obj})
            continue

        for neighbor in graph[current_node]:
            if neighbor in path:
                continue
            distance = graph[current_node][neighbor]
            solar = calculate_solar_exposure(current_node, neighbor)
            new_obj = [obj[0] + distance, obj[1] + solar]
            new_path = path + [neighbor]
            label_id += 1
            heapq.heappush(queue, (sum(new_obj), label_id, new_path, new_obj))

    return paths

# ======================== #result display ========================

results = bi_objective_label_correcting(graph, source, destination)

for i, res in enumerate(results):
    print(f"Path {i+1}: {res['path']}, Distance: {res['objectives'][0]}, Solar: {res['objectives'][1]}")

# ======================== #plot the result ========================

distances = [res['objectives'][0] for res in results]
solar_exposures = [res['objectives'][1] for res in results]

plt.figure(figsize=(6, 4))
plt.scatter(distances, solar_exposures, color='blue')
plt.title("Pareto Frontier: Distance vs Solar Exposure")
plt.xlabel("Distance (m)")
plt.ylabel("Solar Exposure")
plt.grid(True)
plt.show()

In [ ]:
# ======================== #import the package ========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
import os

# ======================== #name the variables ========================
source = (0, 0)
destination = (2, 2)

# ======================== #read the file and store data ========================
# 假设 edge_XX_YY_tree.csv 与 edge_XX_YY_building.csv 全部放在当前工作目录下
corrected_extract_path = "."

# 扫描并读取所有 tree/building CSV，计算每条边的 avg_shadow
# ======================== #read the file and store data ========================
import os
import pandas as pd
import numpy as np

# 1. 自动寻找 CSV 文件所在目录
tree_paths = []
for root, _, files in os.walk('/'):
    for f in files:
        if f.endswith('_tree.csv'):
            tree_paths.append(os.path.join(root, f))
if not tree_paths:
    raise FileNotFoundError("没有找到任何以 '_tree.csv' 结尾的文件，请确认文件已上传。")

# 取第一个找到的目录
corrected_extract_path = os.path.dirname(tree_paths[0])
print("✔️ CSV 文件目录：", corrected_extract_path)

# 2. 扫描该目录下所有 *_tree.csv，成对读取 tree 和 building，计算 avg_shadow 并存入字典
shadow_profiles = {}
for fname in os.listdir(corrected_extract_path):
    if not fname.endswith("_tree.csv"):
        continue
    prefix    = fname[:-9]  # 去掉 "_tree.csv"，得到前缀 "edge_00_01" 等
    tree_file = os.path.join(corrected_extract_path, f"{prefix}_tree.csv")
    bld_file  = os.path.join(corrected_extract_path, f"{prefix}_building.csv")

    if not os.path.exists(bld_file):
        print(f"⚠️ 找不到 {prefix}_building.csv，已跳过")
        continue

    df_tree   = pd.read_csv(tree_file, header=0)
    df_bld    = pd.read_csv(bld_file,  header=0)
    tree_vals = df_tree.select_dtypes(include=[np.number]).astype(float).values
    bld_vals  = df_bld.select_dtypes(include=[np.number]).astype(float).values

    combined   = np.where(bld_vals == 1.0, 1.0, tree_vals)
    avg_shadow = combined.mean(axis=1)

    shadow_profiles[prefix] = avg_shadow

print(f"✔️ 共加载 {len(shadow_profiles)} 条边的遮阴数据")


# ======================== #initialize for the graph ========================
edge_lengths = {
    ((0, 0), (0, 1)): 100, ((0, 1), (0, 2)): 120,
    ((1, 0), (1, 1)): 110, ((1, 1), (1, 2)): 130,
    ((2, 0), (2, 1)): 105, ((2, 1), (2, 2)): 115,
    ((0, 0), (1, 0)): 90,  ((1, 0), (2, 0)): 95,
    ((0, 1), (1, 1)): 85,  ((1, 1), (2, 1)): 100,
    ((0, 2), (1, 2)): 110, ((1, 2), (2, 2)): 105,
    ((0, 0), (1, 1)): 140, ((1, 1), (2, 2)): 145,
    ((0, 2), (1, 1)): 135, ((1, 1), (2, 0)): 150
}

graph = {}
for (i, j), dist in edge_lengths.items():
    graph.setdefault(i, {})[j] = dist
    graph.setdefault(j, {})  # 确保所有节点在字典里

# ======================== #helper functions ========================
def is_dominated(a, b):
    # 检查 a 是否被 b 支配
    return all(x <= y for x, y in zip(a, b)) and any(x < y for x, y in zip(a, b))

def add_label_check(label_dict, node, new_obj):
    # 如果 new_obj 被已有标签支配，则丢弃
    return not any(is_dominated(new_obj, old) for old in label_dict[node])

def calculate_solar_exposure(u, v):
    # 从内存字典中直接取 avg_shadow
    key = f"edge_{u[0]}{u[1]}_{v[0]}{v[1]}"
    shadow_profile = shadow_profiles[key]
    R = np.full_like(shadow_profile, 1000.0)
    C = np.full_like(shadow_profile, 0.2)
    W = (1 - C) * R * (1 - shadow_profile)
    return W.sum()

# ======================== #multi-objective label-correcting 搜索 ========================
def bi_objective_label_correcting(graph, source, destination):
    queue = [(0, 0, [source], [0, 0])]  # (priority, id, path, [dist, solar])
    label_dict = {node: [] for node in graph}
    results = []
    uid = 0

    while queue:
        _, _, path, obj = heapq.heappop(queue)
        u = path[-1]
        if not add_label_check(label_dict, u, obj):
            continue
        label_dict[u].append(obj)

        if u == destination:
            results.append((path, obj))
            continue

        for v, d in graph[u].items():
            if v in path: continue
            solar = calculate_solar_exposure(u, v)
            new_obj = [obj[0] + d, obj[1] + solar]
            uid += 1
            heapq.heappush(queue, (sum(new_obj), uid, path + [v], new_obj))

    return results

# ======================== #run & display ========================
results = bi_objective_label_correcting(graph, source, destination)
for idx, (path, (d, s)) in enumerate(results, 1):
    print(f"Path {idx}: {path}, Distance={d:.1f}, Solar={s:.1f}")

# ======================== #plot Pareto Frontier ========================
ds = [o[1][0] for o in results]
ss = [o[1][1] for o in results]
plt.figure(figsize=(6,4))
plt.scatter(ds, ss)
plt.title("Pareto Frontier")
plt.xlabel("Distance")
plt.ylabel("Solar Exposure")
plt.grid(True)
plt.show()


In [ ]:
# ======================== import ========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq

# ======================== variables ========================
source      = (0, 0)
destination = (2, 2)

# ======================== locate CSV directory ========================
tree_paths = []
for root, _, files in os.walk('/'):
    for f in files:
        if f.endswith('_tree.csv'):
            tree_paths.append(os.path.join(root, f))
if not tree_paths:
    raise FileNotFoundError("未找到任何以 '_tree.csv' 结尾的文件，请确认已上传。")
csv_dir = os.path.dirname(tree_paths[0])

# ======================== load shadow matrices ========================
shadow_profiles = {}
for fname in sorted(os.listdir(csv_dir)):
    if not fname.endswith('_tree.csv'):
        continue
    prefix    = fname[:-9]  # e.g. "edge_00_01"
    tree_file = os.path.join(csv_dir, f"{prefix}_tree.csv")
    bld_file  = os.path.join(csv_dir, f"{prefix}_building.csv")
    if not os.path.exists(bld_file):
        continue

    df_tree = pd.read_csv(tree_file, header=0)
    df_bld  = pd.read_csv(bld_file,  header=0)
    tree_vals = df_tree.select_dtypes(include=[np.number]).values.astype(float)
    bld_vals  = df_bld.select_dtypes(include=[np.number]).values.astype(float)

    combined = np.maximum(tree_vals, bld_vals)  # shape = (T, K)
    shadow_profiles[prefix] = combined

# ======================== graph setup ========================
edge_lengths = {
    ((0, 0), (0, 1)): 100, ((0, 1), (0, 2)): 120,
    ((1, 0), (1, 1)): 110, ((1, 1), (1, 2)): 130,
    ((2, 0), (2, 1)): 105, ((2, 1), (2, 2)): 115,
    ((0, 0), (1, 0)): 90,  ((1, 0), (2, 0)): 95,
    ((0, 1), (1, 1)): 85,  ((1, 1), (2, 1)): 100,
    ((0, 2), (1, 2)): 110, ((1, 2), (2, 2)): 105,
    ((0, 0), (1, 1)): 140, ((1, 1), (2, 2)): 145,
    ((0, 2), (1, 1)): 135, ((1, 1), (2, 0)): 150
}
graph = {}
for (u, v), d in edge_lengths.items():
    graph.setdefault(u, {})[v] = d
    graph.setdefault(v, {})

# ======================== constants ========================
# infer time‐steps from any matrix
T = next(iter(shadow_profiles.values())).shape[0]
# R = np.full(T, 1000.0)
# C = np.full(T,   0.2)

R = np.random.uniform(low=800.0, high=1200.0, size=T)
C = np.random.uniform(low=0.0, high=0.5,    size=T)

# ======================== helper funcs ========================
def is_dominated(a, b):
    return all(x <= y for x,y in zip(a,b)) and any(x < y for x,y in zip(a,b))

def add_label_check(label_dict, node, obj):
    return not any(is_dominated(obj, old) for old in label_dict[node])

def calculate_solar_exposure(u, v, start_time):
    key      = f"edge_{u[0]}{u[1]}_{v[0]}{v[1]}"
    combined = shadow_profiles[key]      # shape=(T, K)
    T_dim, K  = combined.shape
    total = 0.0
    for k in range(K):
        t = start_time + k
        if t >= T: break
        z = combined[t, k]
        if z >= 1:
          z = 1
        total += R[t] * (1 - C[t]) * (1 - z)
    return total

# ======================== multi‐objective search ========================
def bi_objective_label_correcting(graph, source, destination):
    queue      = [(0.0, 0, [source], [0.0, 0.0], 0)]
    label_dict = {n: [] for n in graph}
    results    = []
    uid        = 0

    while queue:
        _, _, path, obj, t0 = heapq.heappop(queue)
        u = path[-1]
        if not add_label_check(label_dict, u, obj):
            continue
        label_dict[u].append(obj)

        if u == destination:
            results.append((path, obj))
            continue

        for v, dist in graph[u].items():
            if v in path: continue
            solar = calculate_solar_exposure(u, v, t0)
            new_obj = [obj[0] + dist, obj[1] + solar]
            # update arrival time
            K = shadow_profiles[f"edge_{u[0]}{u[1]}_{v[0]}{v[1]}"].shape[1]
            new_t0 = t0 + K
            uid += 1
            heapq.heappush(queue, (sum(new_obj), uid, path + [v], new_obj, new_t0))

    return results

# ======================== run search ========================
all_sols = bi_objective_label_correcting(graph, source, destination)

# ======================== filter Pareto front ========================
pareto = []
for path, obj in all_sols:
    if not any(
        (other_obj[0] <= obj[0] and other_obj[1] <= obj[1] and
         (other_obj[0] < obj[0] or other_obj[1] < obj[1]))
        for _, other_obj in all_sols
    ):
        pareto.append((path, obj))

# sort by distance
pareto.sort(key=lambda x: x[1][0])

# ======================== display ========================
print("Pareto‐optimal paths:")
for i, (path, (d, s)) in enumerate(pareto, 1):
    print(f"Path {i}: {path}, Distance={d:.1f}, Solar={s:.1f}")

# plot
ds = [o[1][0] for o in pareto]
ss = [o[1][1] for o in pareto]
plt.figure(figsize=(6,4))
plt.scatter(ds, ss, s=60)
plt.title("Pareto Frontier")
plt.xlabel("Distance (m)")
plt.ylabel("Solar Exposure")
plt.grid(True)
plt.show()


In [ ]:
# ======================== IMPORTS ========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq

# ======================== VARIABLES ========================

# 使用节点ID而非坐标
source_id   = 0
destination_id = 17  # 示例：从节点0到节点17，按需修改

# ======================== LOCATE CSV DIRECTORY ========================
tree_paths = []
for root, _, files in os.walk('/'):
    for f in files:
        if f.endswith('_tree.csv'):
            tree_paths.append(os.path.join(root, f))
if not tree_paths:
    raise FileNotFoundError("未找到任何以 '_tree.csv' 结尾的文件，请确认已上传。")
csv_dir = os.path.dirname(tree_paths[0])

# ======================== LOAD SHADOW PROFILES ========================
shadow_profiles = {}
for fname in sorted(os.listdir(csv_dir)):
    if not fname.endswith('_tree.csv'):
        continue
    prefix      = fname[:-9]  # e.g. 'edge_0_1'
    tree_file   = os.path.join(csv_dir, f"{prefix}_tree.csv")
    shadow_file = os.path.join(csv_dir, f"{prefix}_shadow.csv")
    if not os.path.exists(shadow_file):
        continue
    df_tree   = pd.read_csv(tree_file)
    df_shadow = pd.read_csv(shadow_file)
    tree_vals   = df_tree.select_dtypes(include=[np.number]).values.astype(float)
    shadow_vals = df_shadow.select_dtypes(include=[np.number]).values.astype(float)
    combined = np.maximum(tree_vals, shadow_vals)
    # 存储正反两种前缀，方便无向查找
    shadow_profiles[prefix] = combined
    u, v = prefix.split('_')[1:]  # ['edge','0','1'] → ['0','1']
    rev  = f"edge_{v}_{u}"
    shadow_profiles[rev] = combined

# ======================== NODE COORDS & ID MAPPING ========================
# 仅用于可视化或调试，可不影响核心逻辑
node_coords = {
    0:  ( 0.0,   0.0  ),
    1:  ( 1.0,   0.5  ), 2:  ( 0.5,   1.0  ),  3:  (-0.5,  1.0),
    4:  (-1.0,  0.5  ), 5:  (-1.0, -0.5  ),  6:  (-0.5, -1.0),
    7:  ( 0.5, -1.0  ), 8:  ( 1.0,  -0.5 ),  9:  ( 2.0,   1.0),
    10: ( 1.5,   2.0 ),11: (-1.5,   2.0 ),12: (-2.0,   1.0),
    13: (-2.0,  -1.0 ),14: (-1.5,  -2.0 ),15: ( 1.5,  -2.0),
    16: ( 2.5,   0.0 ),17: ( 2.0,  -1.0 ),18: ( 0.0,   2.0),
    19: ( 0.0,  -2.0 )
}
# 坐标→节点ID 映射（如有必要）
coord_to_id = {coord: nid for nid, coord in node_coords.items()}

# ======================== GRAPH SETUP (USING ID) ========================
# 原本的 coordinate-based 边列表
raw_edges = {
    ((0, 0),   (1, 0.5)):   102, ((0, 0),   (0.5, 1)):    52,
    ((0, 0),   (-0.5, 1)):   48,  ((0, 0),   (-1, 0.5)):   16,
    ((0, 0),   (-1, -0.5)):  43, ((0, 0),   (-0.5, -1)):  25,
    ((0, 0),   (0.5, -1)):   53, ((0, 0),   (1, -0.5)):   66,
    ((1, 0.5), (0.5, 1)):    51, ((0.5, 1), (-0.5, 1)):   93,
    ((-0.5, 1), (-1, 0.5)):  39, ((-1, 0.5), (-1, -0.5)): 43,
    ((-1, -0.5),(-0.5, -1)): 34, ((-0.5, -1), (0.5, -1)): 21,
    ((0.5, -1), (1, -0.5)):  60, ((1, -0.5), (1, 0.5)):   42,
    ((1, 0.5),  (2, 1)):     75, ((0.5, 1),  (0, 2)):     45,
    ((-0.5, 1), (-1.5, 2)):  74, ((-1, 0.5), (-2, 1)):   109,
    ((-1, -0.5),(-2, -1)):   13, ((-0.5, -1),(-1.5, -2)): 54,
    ((0.5, -1),(1.5, -2)):   15, ((1, -0.5), (2, -1)):    63,
    ((2, 1),   (1.5, 2)):    51, ((1.5, 2), (0, 2)):      78,
    ((0, 2),   (-1.5, 2)):   109, ((-1.5, 2),(-2, 1)):     64,
    ((-2, 1),  (-2, -1)):    77, ((-2, -1), (-1.5, -2)):  98,
    ((-1.5, -2),(0, -2)):    95, ((0, -2),  (1.5, -2)):   77,
    ((1.5, -2),(2, -1)):     39, ((2, -1),  (2.5, 0)):    58,
    ((2.5, 0), (2, 1)):      65
}
# 转换为 ID-based graph
graph = {}
for (c_u, c_v), dist in raw_edges.items():
    u_id = coord_to_id[c_u]
    v_id = coord_to_id[c_v]
    if f"edge_{u_id}_{v_id}" in shadow_profiles or f"edge_{v_id}_{u_id}" in shadow_profiles:
        graph.setdefault(u_id, {})[v_id] = dist
        graph.setdefault(v_id, {})[u_id] = dist

# ======================== CONSTANTS ========================
# 时间步长
T = next(iter(shadow_profiles.values())).shape[0]
# 示意用随机 R, C 系数
R = np.random.uniform(800.0, 1200.0, size=T)
C = np.random.uniform(0.0, 0.5, size=T)

# ======================== HELPER FUNCS ========================
def is_dominated(a, b):
    return all(x <= y for x, y in zip(a, b)) and any(x < y for x, y in zip(a, b))

def add_label_check(label_dict, node, obj):
    return not any(is_dominated(obj, old) for old in label_dict[node])

def calculate_solar_exposure(u_id, v_id, start_time):
    key = f"edge_{u_id}_{v_id}"
    if key not in shadow_profiles:
        rev = f"edge_{v_id}_{u_id}"
        if rev in shadow_profiles:
            key = rev
        else:
            raise KeyError(f"Shadow profile not found for edge {u_id}->{v_id}")
    combined = shadow_profiles[key]
    _, K = combined.shape
    total = 0.0
    for k in range(K):
        t = start_time + k
        if t >= T:
            break
        z = min(combined[t, k], 1.0)
        total += R[t] * (1 - C[t]) * (1 - z)
    return total, K

# ======================== MULTI-OBJECTIVE SEARCH ========================
def bi_objective_label_correcting(graph, source, dest):
    queue      = [(0.0, 0, [source], [0.0, 0.0], 0)]
    label_dict = {n: [] for n in graph}
    results    = []
    uid        = 0

    while queue:
        _, _, path, obj, t0 = heapq.heappop(queue)
        u = path[-1]
        if not add_label_check(label_dict, u, obj):
            continue
        label_dict[u].append(obj)
        if u == dest:
            results.append((path, obj))
            continue
        for v, dist in graph[u].items():
            if v in path:
                continue
            solar, steps = calculate_solar_exposure(u, v, t0)
            new_obj = [obj[0] + dist, obj[1] + solar]
            uid += 1
            heapq.heappush(queue, (sum(new_obj), uid, path+[v], new_obj, t0+steps))

    return results

# ======================== RUN & DISPLAY ========================
all_sols = bi_objective_label_correcting(graph, source_id, destination_id)
pareto = [s for s in all_sols if not any(
    (other[1][0] <= s[1][0] and other[1][1] <= s[1][1] and
     (other[1][0] < s[1][0] or other[1][1] < s[1][1]))
    for other in all_sols
)]
pareto.sort(key=lambda x: x[1][0])
print("Pareto-optimal paths:")
for i, (path, (d, s)) in enumerate(pareto, 1):
    print(f"Path {i}: {path}, Distance={d:.1f}, Solar={s:.1f}")
ds = [p[1][0] for p in pareto]; ss = [p[1][1] for p in pareto]
plt.figure(figsize=(6,4)); plt.scatter(ds, ss, s=60)
plt.title("Pareto Frontier"); plt.xlabel("Distance (m)"); plt.ylabel("Solar Exposure")
plt.grid(True); plt.show()


In [ ]:
# ======================== IMPORTS ========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq

# ======================== VARIABLES ========================
# 起点和终点的节点 ID（按需修改）
source_id      = 0
destination_id = 17

# ======================== CSV DIRECTORY ========================
# 假设所有 CSV 文件都在当前工作目录
csv_dir = os.getcwd()

# ======================== LOAD SHADOW PROFILES ========================
shadow_profiles = {}
# 查找所有 tree 文件并加载对应 shadow 文件
for fname in sorted(os.listdir(csv_dir)):
    if not fname.endswith('_tree.csv'):
        continue
    prefix      = fname[:-9]  # 去掉 '_tree.csv'
    tree_file   = os.path.join(csv_dir, f"{prefix}_tree.csv")
    shadow_file = os.path.join(csv_dir, f"{prefix}_shadow.csv")
    if not os.path.exists(shadow_file):
        continue
    # 读取数值矩阵
    df_tree   = pd.read_csv(tree_file)
    df_shadow = pd.read_csv(shadow_file)
    tree_vals   = df_tree.select_dtypes(include=[np.number]).values.astype(float)
    shadow_vals = df_shadow.select_dtypes(include=[np.number]).values.astype(float)
    combined = np.maximum(tree_vals, shadow_vals)
    # 存储正反两种前缀，方便无向查找
    shadow_profiles[prefix] = combined
    u, v = prefix.split('_')[1:]  # 如 ['edge','0','1'] → ['0','1']
    shadow_profiles[f"edge_{v}_{u}"] = combined

# ======================== LOAD SOLAR & CLOUD ========================
# 读取实际时间序列
solar_file = os.path.join(csv_dir, 'solar_intensity.csv')
cloud_file = os.path.join(csv_dir, 'cloud_coverage.csv')
if not os.path.exists(solar_file) or not os.path.exists(cloud_file):
    raise FileNotFoundError("缺少 'solar_intensity.csv' 或 'cloud_coverage.csv' 文件")
solar_df = pd.read_csv(solar_file)
cloud_df = pd.read_csv(cloud_file)
R = solar_df.iloc[:,2].astype(float).values
C = cloud_df.iloc[:,2].astype(float).values

# 时间步长
T = R.shape[0]

# ======================== NODE COORDS & ID MAPPING ========================
node_coords = {
    0: (0, 0),     # Downtown center
    1: (1, 0.5),   # Northeast of downtown
    2: (0.5, 1),   # North of downtown
    3: (-0.5, 1),  # Northwest of downtown
    4: (-1, 0.5),  # West of downtown
    5: (-1, -0.5), # Southwest of downtown
    6: (-0.5, -1), # South of downtown
    7: (0.5, -1),  # Southeast of downtown
    8: (1, -0.5),  # East of downtown
    9: (2, 1),     # Northeast suburb
    10: (1.5, 2),  # North suburb
    11: (-1.5, 2), # Northwest suburb
    12: (-2, 1),   # West suburb
    13: (-2, -1),  # Southwest suburb
    14: (-1.5, -2),# South suburb
    15: (1.5, -2), # Southeast suburb
    16: (2, -1),   # East suburb
    17: (2.5, 0),  # Far east
    18: (0, 2),    # Far north
    19: (0, -2)    # Far south
}

coord_to_id = {coord: nid for nid, coord in node_coords.items()}

# ======================== RAW EDGES (COORDINATE-BASED) ========================
raw_edges = {
    # ——— 从中心 (0,0) 到第一圈 ———
    ((0, 0),   (1, 0.5)):   102,
    ((0, 0),   (0.5, 1)):    52,
    ((0, 0),   (-0.5, 1)):   48,
    ((0, 0),   (-1, 0.5)):   16,
    ((0, 0),   (-1, -0.5)):  43,
    ((0, 0),   (-0.5, -1)):  25,
    ((0, 0),   (0.5, -1)):   53,
    ((0, 0),   (1, -0.5)):   66,

    # ——— 第一圈环状连边 ———
    ((1, 0.5), (0.5, 1)):    51,
    ((0.5, 1), (-0.5, 1)):   93,
    ((-0.5, 1), (-1, 0.5)):  39,
    ((-1, 0.5), (-1, -0.5)): 43,
    ((-1, -0.5),(-0.5, -1)): 34,
    ((-0.5, -1),(0.5, -1)):  21,
    ((0.5, -1), (1, -0.5)):  60,
    ((1, -0.5), (1, 0.5)):   42,

    # ——— 放射到第二圈 ———
    ((1, 0.5),  (2, 1)):     75,   # 1 → 9
    ((0.5, 1),  (0, 2)):     45,   # 2 → 18
    ((-0.5, 1), (-1.5, 2)):  74,   # 3 → 11
    ((-1, 0.5), (-2, 1)):   109,   # 4 → 12
    ((-1, -0.5),(-2, -1)):   13,   # 5 → 13
    ((-0.5, -1),(-1.5, -2)): 54,   # 6 → 14
    ((0.5, -1),(1.5, -2)):   15,   # 7 → 15
    ((1, -0.5), (2, -1)):    63,   # 8 → 17

    # ——— 第二圈环状连边（修正 9,16,17 部分） ———
    ((1.5, 2), (2, 1)):       57,   # 10 ↔  9  （从 51 调整到 57）
    ((2, 1),   (2, -1)):      65,   #  9 ↔ 17  （新增）
    ((2, -1),  (2.5, 0)):     59,   # 17 ↔ 16  （从 58 调整到 59）
    # 注意：删除了原来的 16 ↔  9（(2.5,0)<->(2,1)）这一条

    # ——— 其余第二圈环连边保持不变 ———
    ((1.5, 2), (0, 2)):       78,   # 10 ↔ 18
    ((0, 2),   (-1.5, 2)):   109,   # 18 ↔ 11
    ((-1.5, 2),(-2, 1)):      64,   # 11 ↔ 12
    ((-2, 1),  (-2, -1)):     77,   # 12 ↔ 13
    ((-2, -1), (-1.5, -2)):   98,   # 13 ↔ 14
    ((-1.5, -2),(0, -2)):     95,   # 14 ↔ 19
    ((0, -2),  (1.5, -2)):    77,   # 19 ↔ 15
    ((1.5, -2),(2, -1)):      39    # 15 ↔ 17
}


# ======================== DUMMY SHADOW PROFILE FILL ========================
# 对缺失 shadow profile 的边，用 shape=(T,1) 的低值 dummy 填充
for (cu, cv), _ in raw_edges.items():
    u_id = coord_to_id[cu]
    v_id = coord_to_id[cv]
    p = f"edge_{u_id}_{v_id}"; r = f"edge_{v_id}_{u_id}"
    if p not in shadow_profiles:
        dummy = np.full((T,1), -999999.0)
        shadow_profiles[p] = dummy
        shadow_profiles[r] = dummy

# ======================== GRAPH SETUP (ALL EDGES) ========================
graph = {}
for (cu, cv), dist in raw_edges.items():
    u_id = coord_to_id[cu]; v_id = coord_to_id[cv]
    graph.setdefault(u_id, {})[v_id] = dist
    graph.setdefault(v_id, {})[u_id] = dist

# ======================== HELPER FUNCS ========================
def is_dominated(a, b):
    return all(x <= y for x, y in zip(a, b)) and any(x < y for x, y in zip(a, b))

def add_label_check(label_dict, node, obj):
    return not any(is_dominated(obj, old) for old in label_dict[node])

# ======================== SOLAR EXPOSURE CALC ========================
def calculate_solar_exposure(u_id, v_id, start_time):
    key = f"edge_{u_id}_{v_id}"
    if key not in shadow_profiles:
        key = f"edge_{v_id}_{u_id}"
    profile = shadow_profiles[key]
    _, K = profile.shape
    total = 0.0
    for k in range(K):
        t = start_time + k
        if t >= T: break
        # z < 0 会引入高惩罚：1 - z ≈ 1e6
        z = profile[t, k]
        if z >= 1:
          z = 1
        total += R[t] * (1 - C[t]) * (1 - z)
    return total, K

# ======================== MULTI-OBJECTIVE SEARCH ========================
def bi_objective_label_correcting(graph, source, dest):
    queue      = [(0.0, 0, [source], [0.0, 0.0], 0)]
    label_dict = {n: [] for n in graph}
    results    = []
    uid        = 0

    while queue:
        _, _, path, obj, t0 = heapq.heappop(queue)
        u = path[-1]
        if not add_label_check(label_dict, u, obj):
            continue
        label_dict[u].append(obj)
        if u == dest:
            results.append((path, obj))
            continue
        for v, dist in graph[u].items():
            if v in path: continue
            solar, steps = calculate_solar_exposure(u, v, t0)
            new_obj = [obj[0] + dist, obj[1] + solar]
            uid += 1
            heapq.heappush(queue, (sum(new_obj), uid, path + [v], new_obj, t0 + steps))
    return results

# ======================== RUN & DISPLAY ========================
all_sols = bi_objective_label_correcting(graph, source_id, destination_id)

# 过滤 Pareto
pareto = [s for s in all_sols if not any(
    other[1][0] <= s[1][0] and other[1][1] <= s[1][1] and
    (other[1][0] < s[1][0] or other[1][1] < s[1][1])
    for other in all_sols
)]
pareto.sort(key=lambda x: x[1][0])

print("Pareto-optimal paths:")
for i, (path, (d, s)) in enumerate(pareto, 1):
    print(f"Path {i}: {path}, Distance = {d:.1f}, Solar = {s:.1f}")

# 绘图
ds = [p[1][0] for p in pareto]
ss = [p[1][1] for p in pareto]
plt.figure(figsize=(6, 4))
plt.scatter(ds, ss, s=60)
plt.title("Pareto Frontier: Distance vs Solar Exposure")
plt.xlabel("Distance (m)")
plt.ylabel("Solar Exposure")
plt.grid(True)
plt.show()


In [ ]:
# ======================== IMPORTS ========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
import matplotlib.cm as cm
import matplotlib.patheffects as pe

# ======================== VARIABLES ========================
# 起点和终点的节点 ID（按需修改）
source_id      = 13
destination_id = 17

# ======================== CSV DIRECTORY ========================
# 假设所有 CSV 文件都在当前工作目录
csv_dir = os.getcwd()

# ======================== LOAD SHADOW PROFILES ========================
shadow_profiles = {}
# 查找所有 tree 文件并加载对应 shadow 文件
for fname in sorted(os.listdir(csv_dir)):
    if not fname.endswith('_tree.csv'):
        continue
    prefix      = fname[:-9]  # 去掉 '_tree.csv'
    tree_file   = os.path.join(csv_dir, f"{prefix}_tree.csv")
    shadow_file = os.path.join(csv_dir, f"{prefix}_shadow.csv")
    if not os.path.exists(shadow_file):
        continue
    # 读取数值矩阵
    df_tree   = pd.read_csv(tree_file)
    df_shadow = pd.read_csv(shadow_file)
    tree_vals   = df_tree.select_dtypes(include=[np.number]).values.astype(float)
    shadow_vals = df_shadow.select_dtypes(include=[np.number]).values.astype(float)
    combined = np.maximum(tree_vals, shadow_vals)
    # 存储正反两种前缀，方便无向查找
    shadow_profiles[prefix] = combined
    u, v = prefix.split('_')[1:]  # 如 ['edge','0','1'] → ['0','1']
    shadow_profiles[f"edge_{v}_{u}"] = combined

# ======================== LOAD SOLAR & CLOUD ========================
# 读取实际时间序列
solar_file = os.path.join(csv_dir, 'solar_intensity.csv')
cloud_file = os.path.join(csv_dir, 'cloud_coverage.csv')
if not os.path.exists(solar_file) or not os.path.exists(cloud_file):
    raise FileNotFoundError("缺少 'solar_intensity.csv' 或 'cloud_coverage.csv' 文件")
solar_df = pd.read_csv(solar_file)
cloud_df = pd.read_csv(cloud_file)
R = solar_df.iloc[:,2].astype(float).values
C = cloud_df.iloc[:,2].astype(float).values

# 时间步长
T = R.shape[0]

# ======================== NODE COORDS & ID MAPPING ========================
node_coords = {
    0: (0, 0),         # Downtown center
    1: (0.67, 0.42),   # Northeast of downtown
    2: (0.33, 0.89),   # North of downtown
    3: (-0.48, 0.78),  # Northwest of downtown
    4: (-0.92, 0.31),  # West of downtown
    5: (-0.75, -0.56), # Southwest of downtown
    6: (-0.37, -0.83), # South of downtown
    7: (0.52, -0.69),  # Southeast of downtown
    8: (0.86, -0.27),  # East of downtown
    9: (1.73, 0.86),   # Northeast suburb
    10: (1.26, 1.87),  # North suburb
    11: (-1.32, 1.76), # Northwest suburb
    12: (-1.89, 0.72), # West suburb
    13: (-1.65, -0.94),# Southwest suburb
    14: (-1.18, -1.73),# South suburb
    15: (1.31, -1.62), # Southeast suburb
    16: (1.81, -0.83), # East suburb
    17: (2.23, 0.21),  # Far east
    18: (0.18, 1.76),  # Far north
    19: (0.11, -1.65)  # Far south
}

coord_to_id = {coord: nid for nid, coord in node_coords.items()}

# ======================== RAW EDGES (COORDINATE-BASED) ========================
raw_edges = {
    # ——— from center to first ring ———
    ((0, 0),       (0.67, 0.42)):     40,
    ((0, 0),       (0.33, 0.89)):     55,
    ((0, 0),       (-0.48, 0.78)):    58,
    ((0, 0),       (-0.92, 0.31)):    54,
    ((0, 0),       (-0.75, -0.56)):   52,
    ((0, 0),       (-0.37, -0.83)):   57,
    ((0, 0),       (0.52, -0.69)):    53,
    ((0, 0),       (0.86, -0.27)):    57,

    # ——— first-ring cycle ———
    ((0.67, 0.42), (0.33, 0.89)):     27,
    ((0.33, 0.89), (-0.48, 0.78)):    45,
    ((-0.48, 0.78),(-0.92, 0.31)):    35,
    ((-0.92, 0.31),(-0.75, -0.56)):   54,
    ((-0.75, -0.56),(-0.37, -0.83)):  20,
    ((-0.37, -0.83),(0.52, -0.69)):   54,
    ((0.52, -0.69),(0.86, -0.27)):    26,
    ((0.86, -0.27),(0.67, 0.42)):     42,

    # ——— spokes to second ring ———
    ((0.67, 0.42),  (1.73, 0.86)):    73,   # 1 → 9
    ((0.33, 0.89),  (0.18, 1.76)):    56,   # 2 → 18
    ((-0.48, 0.78), (-1.32, 1.76)):   87,   # 3 → 11
    ((-0.92, 0.31), (-1.89, 0.72)):   59,   # 4 → 12
    ((-0.75, -0.56),(-1.65, -0.94)):  54,   # 5 → 13
    ((-0.37, -0.83),(-1.18, -1.73)):  80,   # 6 → 14
    ((0.52, -0.69), (1.31, -1.62)):   71,   # 7 → 15
    ((0.86, -0.27), (1.81, -0.83)):   70,   # 8 → 16

    # ——— second-ring cycle (adjusted 9–16–17) ———
    ((1.26, 1.87),  (1.73, 0.86)):    74,   # 10 ↔ 9
    ((1.73, 0.86),  (2.23, 0.21)):    46,   # 9 ↔ 17
    ((1.81, -0.83), (2.23, 0.21)):    72,   # 16 ↔ 17
    # (removed original 17 ↔ 9 link)

    # ——— remaining second-ring connections ———
    ((1.26, 1.87),  (0.18, 1.76)):    67,   # 10 ↔ 18
    ((0.18, 1.76),  (-1.32, 1.76)):   89,   # 18 ↔ 11
    ((-1.32, 1.76), (-1.89, 0.72)):   75,   # 11 ↔ 12
    ((-1.89, 0.72), (-1.65, -0.94)):  110,   # 12 ↔ 13
    ((-1.65, -0.94),(-1.18, -1.73)):  55,   # 13 ↔ 14
    ((-1.18, -1.73),(0.11, -1.65)):   74,   # 14 ↔ 19
    ((0.11, -1.65), (1.31, -1.62)):   79,   # 19 ↔ 15
    ((1.31, -1.62), (1.81, -0.83)):   56    # 15 ↔ 16
}



# ======================== DUMMY SHADOW PROFILE FILL ========================
# 对缺失 shadow profile 的边，用 shape=(T,1) 的低值 dummy 填充
for (cu, cv), _ in raw_edges.items():
    u_id = coord_to_id[cu]
    v_id = coord_to_id[cv]
    p = f"edge_{u_id}_{v_id}"; r = f"edge_{v_id}_{u_id}"
    if p not in shadow_profiles:
        dummy = np.full((T,1), -999999.0)
        shadow_profiles[p] = dummy
        shadow_profiles[r] = dummy

# ======================== GRAPH SETUP (ALL EDGES) ========================
graph = {}
for (cu, cv), dist in raw_edges.items():
    u_id = coord_to_id[cu]; v_id = coord_to_id[cv]
    graph.setdefault(u_id, {})[v_id] = dist
    graph.setdefault(v_id, {})[u_id] = dist

# ======================== HELPER FUNCS ========================
def is_dominated(a, b):
    return all(x <= y for x, y in zip(a, b)) and any(x < y for x, y in zip(a, b))

def add_label_check(label_dict, node, obj):
    return not any(is_dominated(obj, old) for old in label_dict[node])

# ======================== SOLAR EXPOSURE CALC ========================
def calculate_solar_exposure(u_id, v_id, start_time):
    key = f"edge_{u_id}_{v_id}"
    if key not in shadow_profiles:
        key = f"edge_{v_id}_{u_id}"
    profile = shadow_profiles[key]
    _, K = profile.shape
    total = 0.0
    for k in range(K):
        t = start_time + k
        if t >= T: break
        # z < 0 会引入高惩罚：1 - z ≈ 1e6
        z = profile[t, k]
        if z >= 1:
          z = 1
        total += R[t] * (1 - C[t]) * (1 - z)
    return total, K

# ======================== MULTI-OBJECTIVE SEARCH ========================
def bi_objective_label_correcting(graph, source, dest):
    queue      = [(0.0, 0, [source], [0.0, 0.0], 0)]
    label_dict = {n: [] for n in graph}
    results    = []
    uid        = 0

    while queue:
        _, _, path, obj, t0 = heapq.heappop(queue)
        u = path[-1]
        if not add_label_check(label_dict, u, obj):
            continue
        label_dict[u].append(obj)
        if u == dest:
            results.append((path, obj))
            continue
        for v, dist in graph[u].items():
            if v in path: continue
            solar, steps = calculate_solar_exposure(u, v, t0)
            new_obj = [obj[0] + dist, obj[1] + solar]
            uid += 1
            heapq.heappush(queue, (sum(new_obj), uid, path + [v], new_obj, t0 + steps))
    return results

# ======================== RUN & DISPLAY ========================
all_sols = bi_objective_label_correcting(graph, source_id, destination_id)

In [ ]:

# ======================== PARETO FRONTIER PLOT (美化) ========================
# 过滤 Pareto
pareto = [s for s in all_sols if not any(
    other[1][0] <= s[1][0] and other[1][1] <= s[1][1] and
    (other[1][0] < s[1][0] or other[1][1] < s[1][1])
    for other in all_sols
)]
# ——— Extract the two objective lists ———
ds = [path_obj[1][0] for path_obj in pareto]   # total distances
ss = [path_obj[1][1] for path_obj in pareto]   # total solar exposures

# 1) 计算均值
mean_d = sum(ds) / len(ds)
mean_s = sum(ss) / len(ss)

# 2) 找到三个特殊点索引
idx_shortest   = min(range(len(ds)), key=lambda i: ds[i])
idx_best_shade = min(range(len(ss)), key=lambda i: ss[i])
idx_compromise = min(
    range(len(ds)),
    key=lambda i: (ds[i] - mean_d)**2 + (ss[i] - mean_s)**2
)

# 3) 开始绘图
plt.figure(figsize=(10, 6))

# 所有 Pareto 点
plt.scatter(ds, ss,
            marker='x',
            s=400,
            color='tab:blue',
            label='Pareto-optimal paths')

# 高亮设置：索引, 标签, 颜色, 垂直对齐, 水平对齐
highlights = [
    (idx_shortest,  'Shortest Path',        'tab:green',  'bottom', 'left'),
    (idx_best_shade, 'Best Shade',           'tab:orange', 'top',    'right'),
    (idx_compromise, 'Balanced Compromise',  'tab:red',    'bottom', 'right'),
]

for idx, label, color, va, ha in highlights:
    plt.scatter(ds[idx], ss[idx],
                marker='X',
                s=800,
                color=color,
                linewidths=0.8,   # 控制 “×” 的粗细，数值越小越细
                label=label)

# 标题与坐标轴
plt.title("Pareto Frontier: Distance vs. Solar Exposure", fontsize=16, pad=12)
plt.xlabel("Total Distance", fontsize=14)
plt.ylabel("Total Solar Exposure", fontsize=14)

# 网格
plt.grid(linestyle='--', alpha=0.5)

# 美观的图例：放图内右上角，略偏移避免重叠
leg = plt.legend(
    loc='upper right',        # 图例位置
    fontsize=16,              # 文字大小
    frameon=True,             # 显示边框
    borderpad=1.0,            # 边框内间距
    labelspacing=1.0,         # 条目之间间隔
    handlelength=2.5,         # 图例句柄长度
    handletextpad=1.0,        # 图例句柄与文字间距
    markerscale=0.8           # 放大标记尺寸
)
leg.get_frame().set_edgecolor('gray')   # 边框颜色
leg.get_frame().set_alpha(0.9)          # 背景透明度


# 留出图例空间
plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.show()

In [ ]:

# ======================== PARETO FRONTIER PLOT (美化) ========================
# 过滤 Pareto
pareto = [s for s in all_sols if not any(
    other[1][0] <= s[1][0] and other[1][1] <= s[1][1] and
    (other[1][0] < s[1][0] or other[1][1] < s[1][1])
    for other in all_sols
)]
# ——— Extract the two objective lists ———
ds = [path_obj[1][0] for path_obj in pareto]   # total distances
ss = [path_obj[1][1] for path_obj in pareto]   # total solar exposures

# 1) 计算均值
mean_d = sum(ds) / len(ds)
mean_s = sum(ss) / len(ss)

# 2) 找到三个特殊点索引
idx_shortest   = min(range(len(ds)), key=lambda i: ds[i])
idx_best_shade = min(range(len(ss)), key=lambda i: ss[i])
idx_compromise = min(
    range(len(ds)),
    key=lambda i: (ds[i] - mean_d)**2 + (ss[i] - mean_s)**2
)

# 3) 开始绘图
plt.figure(figsize=(10, 6))

# 所有 Pareto 点
plt.scatter(ds, ss,
            marker='x',
            s=400,
            color='tab:blue',
            label='Pareto-optimal paths')

# 高亮设置：索引, 标签, 颜色, 垂直对齐, 水平对齐
highlights = [
    (idx_shortest,  'Shortest Path',        'tab:green',  'bottom', 'left'),
    (idx_best_shade, 'Best Shade',           'tab:orange', 'top',    'right'),
    (idx_compromise, 'Balanced Compromise',  'tab:red',    'bottom', 'right'),
]

for idx, label, color, va, ha in highlights:
    plt.scatter(ds[idx], ss[idx],
                marker='X',
                s=800,
                color=color,
                linewidths=0.8,   # 控制 “×” 的粗细，数值越小越细
                label=label)

# 标题与坐标轴
plt.title("Pareto Frontier: Distance vs. Solar Exposure", fontsize=16, pad=12)
plt.xlabel("Total Distance", fontsize=14)
plt.ylabel("Total Solar Exposure", fontsize=14)

# 网格
plt.grid(linestyle='--', alpha=0.5)

# 美观的图例：放图内右上角，略偏移避免重叠
leg = plt.legend(
    loc='upper right',        # 图例位置
    fontsize=16,              # 文字大小
    frameon=True,             # 显示边框
    borderpad=1.0,            # 边框内间距
    labelspacing=1.0,         # 条目之间间隔
    handlelength=2.5,         # 图例句柄长度
    handletextpad=1.0,        # 图例句柄与文字间距
    markerscale=0.8           # 放大标记尺寸
)
leg.get_frame().set_edgecolor('gray')   # 边框颜色
leg.get_frame().set_alpha(0.9)          # 背景透明度


# 留出图例空间
plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.show()

In [ ]:
# ======================== DRAW & ROUTE ========================
# 假定 graph, node_coords, pareto 都已定义

fig, ax = plt.subplots(figsize=(14, 12))

# 1) 背景路段（浅灰色）
seen = set()
for u, nbrs in graph.items():
    for v, dist in nbrs.items():
        if (v, u) in seen: continue
        seen.add((u, v))
        x1, y1 = node_coords[u]
        x2, y2 = node_coords[v]
        ax.plot([x1, x2], [y1, y2],
                color='lightgray', linewidth=1, zorder=1)

# 2) 节点：先画一个白底黑边的大点，再加粗数字
for nid, (x, y) in node_coords.items():
    # 用黄色实心圆 + 黑边
    ax.scatter(x, y,
               s=300,                # 点更大
               facecolor='pink',   # 底色改为黄色
               edgecolor='black',    # 黑边
               linewidth=2,
               zorder=2)
    # 编号文字略微向上偏移，用描边让文字更易读
    txt = ax.text(x, y + 0.1, str(nid),
                  ha='center', va='bottom',
                  fontsize=14,
                  fontweight='bold',
                  color='black',
                  zorder=3)
    # 添加白色描边（halo）
    txt.set_path_effects([
        pe.Stroke(linewidth=3, foreground='white'),
        pe.Normal()
    ])

# 3) Pareto 路径：不同线型 + 半透明 + 颜色
cmap = cm.get_cmap('tab10', len(pareto))
line_styles = ['solid', 'dashed', 'dotted', 'dashdot']
for i, (path, (dist, sol)) in enumerate(pareto):
    pts = [node_coords[n] for n in path]
    xs, ys = zip(*pts)
    ax.plot(xs, ys,
            linestyle=line_styles[i % len(line_styles)],
            marker='o',
            markersize=8,
            linewidth=2.5,
            alpha=0.9,
            color=cmap(i),
            label=f'Path {i+1}: d={dist:.0f}, s={sol:.1f}',
            zorder=4)

# 4) 图例移动到画布外，并优化布局
ax.legend(loc='upper left',
          bbox_to_anchor=(1.02, 1),
          fontsize='small',
          frameon=True)

# 美化
ax.set_title("Pareto-optimal Paths on Pedestrian Network", fontsize=16)
ax.set_xlabel("X Coordinate", fontsize=14)
ax.set_ylabel("Y Coordinate", fontsize=14)
ax.set_aspect('equal', 'box')
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
ax = plt.gca()

for (u, v), shade in edge_shade.items():
    if u < v:
        x1, y1 = node_coords[u]
        x2, y2 = node_coords[v]
        ax.plot([x1, x2], [y1, y2],
                linewidth=8,
                solid_capstyle='butt',
                color=plt.cm.YlGn(shade),  # 0→yellow, 1→green
                alpha=0.8)

# 节点
xs, ys = zip(*node_coords.values())
ax.scatter(xs, ys, s=50, color='white', edgecolor='black', zorder=2)

for nid, (x, y) in node_coords.items():
    # 用黄色实心圆 + 黑边
    ax.scatter(x, y,
               s=300,                # 点更大
               facecolor='pink',   # 底色改为黄色
               edgecolor='black',    # 黑边
               linewidth=2,
               zorder=2)
    # 编号文字略微向上偏移，用描边让文字更易读
    txt = ax.text(x, y + 0.1, str(nid),
                  ha='center', va='bottom',
                  fontsize=14,
                  fontweight='bold',
                  color='black',
                  zorder=3)
    # 添加白色描边（halo）
    txt.set_path_effects([
        pe.Stroke(linewidth=3, foreground='white'),
        pe.Normal()
    ])

# 色标
sm = plt.cm.ScalarMappable(cmap='YlGn', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Average Shade (0=no, 1=full)', fontsize=12)

ax.set_aspect('equal')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Road Segment Shade Heatmap", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

# 假设 raw_edges, node_coords, shadow_profiles, coord_to_id, graph 等变量
# 已经在脚本中定义并计算了 edge_shade（每条边的平均 shade）

# 1) 计算每个节点的平均 shade 值
node_shade = {nid: [] for nid in node_coords}
for (u, v), shade in edge_shade.items():
    node_shade[u].append(shade)
    node_shade[v].append(shade)
# 只保留平均值
node_vals = {nid: np.mean(vals) for nid, vals in node_shade.items()}

# 2) 准备三角剖分输入
xs = np.array([node_coords[n][0] for n in node_coords])
ys = np.array([node_coords[n][1] for n in node_coords])
zs = np.array([node_vals[n] for n in node_coords])

triang = mtri.Triangulation(xs, ys)

# 3) 绘制等高线填色图
plt.figure(figsize=(10, 8))
contf = plt.tricontourf(triang, zs, levels=12, cmap='YlGn', alpha=0.8)
cont = plt.tricontour(triang, zs, levels=12, colors='k', linewidths=0.5)

# 4) 叠加网络边
for u, nbrs in graph.items():
    for v in nbrs:
        if u < v:
            x1, y1 = node_coords[u]
            x2, y2 = node_coords[v]
            plt.plot([x1, x2], [y1, y2], color='gray', linewidth=1, alpha=0.6)

# 5) 叠加节点
for nid, (x, y) in node_coords.items():
    plt.scatter(x, y, s=40, facecolor='white', edgecolor='black', linewidth=0.8, zorder=3)
    plt.text(x, y+0.02, str(nid), ha='center', va='bottom', fontsize=9)

plt.colorbar(contf, label='Average Shade (0=no, 1=full)')
plt.title("Shadow Intensity Contour Map")
plt.axis('equal')
plt.xticks([]); plt.yticks([])
plt.tight_layout()
plt.show()


In [ ]:
# …（前面数据准备完毕）…

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 光滑曲面，cmap='YlGn'：0→yellow，1→green
surf = ax.plot_surface(
    Xi, Yi, Zi,
    cmap='YlGn',
    edgecolor='none',
    antialiased=True,
    rcount=10000, ccount=10000  # 更细网格
)

# 原始节点叠加
ax.scatter(xs, ys, zs, color='black', s=30, zorder=5)

# **调整视角**：抬高视点，增大俯仰，让曲面更“倾斜”
ax.view_init(elev=45, azim=-60)

# 坐标轴标签和标题
ax.set_title("Smooth 3D Shadow Surface", fontsize=16)
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.set_zlabel("Average Shade (0–1)")

# 色标
cbar = fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Average Shade')
plt.tight_layout()
plt.show()
